Baseline and model benchmarking 

In [1]:
# 1. imports and setup

import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
import lightgbm as lgb

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
print(f"Project Root Verified: {ROOT.resolve()}")

Project Root Verified: D:\DepRadar


1. temporal data prep with 6 month embargo

training snapshot T=2023-01-02 and validation snapshot T=2023-08-28

In [3]:
train_date = '2023-01-02'
val_date = '2023-08-28'

df_train_feat = pd.read_parquet(ROOT / 'data' / 'processed' / f'T={train_date}' / 'features.parquet')
df_train_lbl = pd.read_parquet(ROOT / 'data' / 'processed' / f'T={train_date}' / 'labels.parquet')

df_val_feat = pd.read_parquet(ROOT / 'data' / 'processed' / f'T={val_date}' / 'features.parquet')
df_val_lbl = pd.read_parquet(ROOT / 'data' / 'processed' / f'T={val_date}' / 'labels.parquet')

y_train = df_train_lbl['label'].values
y_val = df_val_lbl['label'].values

# Sub-group mask: packages with ZERO prior advisories at validation cutoff
first_time_mask = (df_val_lbl['had_prior_advisory'] == 0).values

print(f"Train Snapshot ({train_date}): {len(df_train_feat):,} packages | Positives: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"Val Snapshot   ({val_date}):   {len(df_val_feat):,} packages | Positives: {y_val.sum():,} ({y_val.mean()*100:.2f}%)")
print(f"Validation First-Time Positives: {y_val[first_time_mask].sum():,} / {y_val.sum():,} ({y_val[first_time_mask].sum()/y_val.sum()*100:.1f}%)")

Train Snapshot (2023-01-02): 30,000 packages | Positives: 95 (0.32%)
Val Snapshot   (2023-08-28):   30,000 packages | Positives: 82 (0.27%)
Validation First-Time Positives: 41 / 82 (50.0%)


2. Feature scopes and preprocessing 

there will be two feature sets 
- tablular features
- graph features

In [4]:
# Define feature subsets
TABULAR_FEATURES = [
    'age_days', 'n_versions', 'n_release_versions', 'days_since_last_release',
    'releases_90d', 'releases_365d', 'median_gap_days_last20', 'n_deps_prod',
    'log_dependents', 'dependents_delta_3m', 'has_repo', 'log_stars', 'log_forks',
    'n_prior_adv', 'n_prior_adv_365d', 'days_since_last_adv', 'has_prior_adv',
    'max_severity_ord', 'n_prior_critical_high'
]

GRAPH_FEATURES = [
    'in_deg', 'out_deg', 'nb_out_mean_prior_adv', 'nb_out_max_prior_adv',
    'nb_out_frac_prior_adv', 'nb_in_mean_prior_adv', 'nb_in_max_prior_adv',
    'nb_in_frac_prior_adv', 'nb_out_mean_log_dependents', 'nb_in_mean_log_dependents'
]

ALL_FEATURES = TABULAR_FEATURES + GRAPH_FEATURES

# build matrices 
X_train_tab = df_train_feat[TABULAR_FEATURES].fillna(0).values
X_val_tab = df_val_feat[TABULAR_FEATURES].fillna(0).values

X_train_all = df_train_feat[ALL_FEATURES].fillna(0).values
X_val_all = df_val_feat[ALL_FEATURES].fillna(0).values

# standardized matrices for linear model
scaler  = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_all)
X_val_scaled   = scaler.transform(X_val_all)

print(f"Tabular Predictors: {len(TABULAR_FEATURES)} | Full Predictors (+Graph): {len(ALL_FEATURES)}")

Tabular Predictors: 19 | Full Predictors (+Graph): 29


3. Evaluation Metric functions 

function to have metrics scores easily

In [6]:
# evaluation helper function
def evaluate_predictions(y_true, scores, k=500):
    roc = roc_auc_score(y_true, scores)
    pr  = average_precision_score(y_true, scores)
    brier = brier_score_loss(y_true, scores)
    
    # Top-K ranking metrics
    ranked_idx = np.argsort(scores)[::-1]
    top_k_idx  = ranked_idx[:k]
    hits       = y_true[top_k_idx].sum()
    total_pos  = y_true.sum()
    
    recall_at_k = (hits / max(total_pos, 1)) * 100
    precision_at_k = (hits / k) * 100
    base_rate = (total_pos / len(y_true)) * 100
    lift_at_k = precision_at_k / max(base_rate, 1e-6)
    
    return {
        'ROC-AUC': roc,
        'PR-AUC': pr,
        f'Recall@{k} (%)': recall_at_k,
        f'Precision@{k} (%)': precision_at_k,
        f'Lift@{k} (x)': lift_at_k,
        'Brier Score': brier
    }
